# Data Profiling e Integración de Fuentes para Clustering de Empresas

## Objetivo

Este cuaderno documenta el proceso de análisis, validación e integración de dos fuentes de datos empresariales con el objetivo de construir un dataset adecuado para técnicas de clustering.

## Fuentes de datos

- Dataset 1: Leads provenientes de CRM
- Dataset 2: Registro de horas trabajadas por empresa

## Enfoque

Se sigue una metodología basada en:

1. Data profiling
2. Validación de hipótesis de integración
3. Evaluación de consistencia entre fuentes
4. Toma de decisiones basada en evidencia
5. Recomendación de siguiente paso (staging y estrategia de integración)

## 1. Imports y configuración


In [1]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)


## 2. Configuración de archivos


In [2]:
LEADS_FILE = 'leads.xlsx'
HORAS_FILE = 'proyectos_empresa.xlsx'

N_FILAS_PRUEBA = None


## 3. Función de carga de datos


In [3]:
def leer_archivo(path, nrows=None):
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == '.csv':
        return pd.read_csv(path, nrows=nrows)
    elif suffix in ['.xlsx', '.xls']:
        return pd.read_excel(path, nrows=nrows)
    else:
        raise ValueError(f'Formato no soportado: {suffix}')


## 4. Carga de datasets


In [4]:
df_leads = leer_archivo(LEADS_FILE, nrows=N_FILAS_PRUEBA)
df_horas = leer_archivo(HORAS_FILE, nrows=N_FILAS_PRUEBA)

print('Leads:', df_leads.shape)
print('Horas:', df_horas.shape)


Leads: (440, 15)
Horas: (412, 18)


C:\Users\asus\AppData\Roaming\Python\Python312\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


## 4.b Perfilado rápido (calidad y esquema)


In [5]:
def _profile_table(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame({
        'columna': df.columns,
        'dtype': [str(df[c].dtype) for c in df.columns],
        'n_null': [int(df[c].isna().sum()) for c in df.columns],
        'pct_null': [float(df[c].isna().mean() * 100) for c in df.columns],
        'n_unique': [int(df[c].nunique(dropna=True)) for c in df.columns],
    })
    return out.sort_values(['pct_null', 'n_unique'], ascending=[False, True]).reset_index(drop=True)

def _normalize_for_compare(value) -> str:
    if value is None:
        return ''
    if isinstance(value, float) and np.isnan(value):
        return ''
    s = str(value).strip()
    if not s:
        return ''
    s = unicodedata.normalize('NFKD', s)
    s = ''.join(ch for ch in s if not unicodedata.combining(ch))
    s = s.upper()
    s = re.sub(r'[^A-Z0-9 ]+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def _company_quality(df: pd.DataFrame, col: str, label: str) -> None:
    if col not in df.columns:
        print(f'[{label}] Columna no encontrada: {col}')
        return
    s = df[col]
    s_str = s.astype('string')
    blank = s_str.isna() | (s_str.str.strip() == '')
    print(f'\n[{label}] Calidad de {col}:')
    print('- filas:', len(df))
    print('- n_null:', int(s.isna().sum()))
    print('- n_blank (null o vacío):', int(blank.sum()))
    print('- n_unique (no-null):', int(s.nunique(dropna=True)))
    print('- n_duplicadas (por valor no-null):', int(s.dropna().duplicated().sum()))

    top = (s_str.fillna('')
              .map(lambda x: x.strip())
              .replace('', pd.NA)
              .dropna()
              .value_counts()
              .head(15)
              .reset_index())
    if len(top):
        top.columns = [col, 'freq']
        display(top)

    # Normalización rápida para estimar colisiones
    norm = s_str.map(_normalize_for_compare)
    norm = norm.replace('', pd.NA).dropna()
    if len(norm):
        print('- n_unique_norm:', int(norm.nunique()))
        print('- colisiones_por_norm (valores distintos que caen al mismo norm):', int(norm.duplicated().sum()))

def _exact_match_summary() -> None:
    if 'Company' not in df_leads.columns or 'EMPRESA' not in df_horas.columns:
        print('\n[JOIN] No se puede calcular resumen: falta Company o EMPRESA.')
        return

    leads_raw = (df_leads['Company'].astype('string').fillna('').map(lambda x: x.strip()))
    horas_raw = (df_horas['EMPRESA'].astype('string').fillna('').map(lambda x: x.strip()))
    leads_raw = leads_raw.replace('', pd.NA).dropna()
    horas_raw = horas_raw.replace('', pd.NA).dropna()

    raw_matches = sorted(set(leads_raw).intersection(set(horas_raw)))
    print('\n[JOIN] Coincidencias exactas (sin normalizar):', len(raw_matches))
    if len(raw_matches):
        print('  Muestra:', raw_matches[:20])

    leads_norm = leads_raw.map(_normalize_for_compare).replace('', pd.NA).dropna()
    horas_norm = horas_raw.map(_normalize_for_compare).replace('', pd.NA).dropna()
    norm_matches = sorted(set(leads_norm).intersection(set(horas_norm)))
    print('[JOIN] Coincidencias exactas (normalizadas):', len(norm_matches))
    if len(norm_matches):
        print('  Muestra:', norm_matches[:20])

print('--- Perfilado LEADS ---')
display(_profile_table(df_leads))

print('\n--- Perfilado HORAS ---')
display(_profile_table(df_horas))

# Calidad de las llaves candidatas para join
_company_quality(df_leads, 'Company', 'LEADS')
_company_quality(df_horas, 'EMPRESA', 'HORAS')

# Evidencia cuantitativa: ¿hay intersección exacta?
_exact_match_summary()

--- Perfilado LEADS ---


,columna,dtype,n_null,pct_null,n_unique
0,First Name,float64,440,100.000000,0
1,Last Name,float64,440,100.000000,0
2,Email,float64,440,100.000000,0
3,Phone,float64,440,100.000000,0
4,Mobile,float64,440,100.000000,0
5,Website,float64,440,100.000000,0
6,No. of Employees,float64,440,100.000000,0
7,Annual Revenue,float64,440,100.000000,0
8,Linkedin,float64,440,100.000000,0
9,País.,object,415,94.318182,8



--- Perfilado HORAS ---


,columna,dtype,n_null,pct_null,n_unique
0,AVANCE_REAL,float64,412,100.000000,0
1,AVANCE_ESTIMADO,float64,412,100.000000,0
2,ID_COL_RESPONSABLE,float64,379,91.990291,7
3,FACTURACION,float64,235,57.038835,120
4,HORAS_ESTIMADAS,float64,187,45.388350,107
5,HORAS_EJECUTADAS_FACTURABLES,float64,26,6.310680,314
6,HORAS_EJECUTADAS,float64,20,4.854369,319
7,FECHA_CORTE,datetime64[ns],19,4.611650,5
8,EN_EJECUCION,float64,4,0.970874,2
9,MOSTRAR_LISTAS,int64,0,0.000000,2



[LEADS] Calidad de Company:
- filas: 440
- n_null: 0
- n_blank (null o vacío): 0
- n_unique (no-null): 330
- n_duplicadas (por valor no-null): 110


,Company,freq
0,GRUPO ALEN,10
1,PEPSICO,7
2,GRUPO BIMBO,7
3,WALMART,7
4,CASA CUERVO,5
5,BACARDI,5
6,SEGUROS MONTERREY NEW YORK LIFE,5
7,LAMOSA,5
8,UNILEVER,4
9,AIG,4


- n_unique_norm: 328
- colisiones_por_norm (valores distintos que caen al mismo norm): 112

[HORAS] Calidad de EMPRESA:
- filas: 412
- n_null: 0
- n_blank (null o vacío): 0
- n_unique (no-null): 120
- n_duplicadas (por valor no-null): 292


,EMPRESA,freq
0,Corporacion GPF,66
1,Corporación Maresa,22
2,FPA,21
3,Veolia Latam,21
4,Aseguradora del Sur,20
5,Intaco,12
6,La Fabril,10
7,Telefónica EC,10
8,NOVA Ecuador,10
9,"Millicom - Telefónica PA, NI",9


- n_unique_norm: 119
- colisiones_por_norm (valores distintos que caen al mismo norm): 293

[JOIN] Coincidencias exactas (sin normalizar): 0
[JOIN] Coincidencias exactas (normalizadas): 0


## 5. Validación de hipótesis de integración

### Hipótesis

Se plantea que:

> Las empresas presentes en el dataset de leads deberían coincidir con las empresas presentes en el dataset de horas trabajadas.

### Objetivo

Validar si es posible realizar un join confiable entre ambas fuentes.


## 6. Evaluación detallada (coincidencias exactas normalizadas y aproximadas)

El resumen de la sección **4.b (Perfilado rápido)** puede ampliarse con una rutina más detallada de normalización y similitud.
Esta sección se conserva porque aporta evidencia técnica adicional:

- normaliza nombres con eliminación de acentos,
- reduce ruido por sufijos legales comunes,
- calcula coincidencias exactas después de normalizar,
- y genera sugerencias por similitud para evidenciar que no existen matches confiables.

Esto respalda formalmente la decisión de **no forzar un join** entre ambas fuentes usando solo el nombre de la empresa.

In [6]:
import re
import unicodedata
import pandas as pd

def _strip_accents(s: str) -> str:
    return ''.join(ch for ch in unicodedata.normalize('NFKD', s) if not unicodedata.combining(ch))

def _normalize_name(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip()
    s = _strip_accents(s)
    s = s.upper()
    # quitar caracteres raros, dejar letras/números/espacios
    s = re.sub(r"[^A-Z0-9 ]+", " ", s)
    # quitar sufijos legales comunes
    s = re.sub(r"\b(SA|S A|S\.A|S\.A\.S|SAS|LTDA|CIA|CORP|INC|LLC|DE|DEL|LA|EL)\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Tomar listas directamente desde los DataFrames (evita dependencia de secciones eliminadas)
if 'Company' not in df_leads.columns or 'EMPRESA' not in df_horas.columns:
    print("No se puede comparar: falta Company o EMPRESA en los dataframes.")
else:
    companies = (df_leads['Company'].astype('string').fillna('')
                .map(lambda x: x.strip())
                .replace('', pd.NA)
                .dropna()
                .unique()
                .tolist())
    empresas = (df_horas['EMPRESA'].astype('string').fillna('')
                .map(lambda x: x.strip())
                .replace('', pd.NA)
                .dropna()
                .unique()
                .tolist())

    if not companies or not empresas:
        print("No hay valores suficientes para comparar (listas vacías).")
    else:
        comp_norm = {c: _normalize_name(c) for c in companies}
        emp_norm = {e: _normalize_name(e) for e in empresas}

        # 1) Coincidencias exactas después de normalizar
        inv_emp = {}
        for e, ne in emp_norm.items():
            inv_emp.setdefault(ne, []).append(e)

        exact_matches = []
        for c, nc in comp_norm.items():
            for e in inv_emp.get(nc, []):
                exact_matches.append({
                    "Company": c,
                    "EMPRESA": e,
                    "tipo": "exact_norm",
                    "score": 100
                })

        df_exact = pd.DataFrame(exact_matches)
        print(f"Coincidencias EXACTAS (tras normalizar): {len(df_exact)}")
        if len(df_exact):
            display(df_exact.sort_values(['Company', 'EMPRESA']).reset_index(drop=True))

        # 2) Coincidencias por similitud (fuzzy). Si está rapidfuzz, mejor; si no, usa difflib.
        rows = []
        try:
            from rapidfuzz import process, fuzz
            scorer = fuzz.token_set_ratio
            emp_choices = list(emp_norm.keys())
            for c, nc in comp_norm.items():
                best = process.extractOne(nc, emp_choices, scorer=scorer)
                if best is None:
                    continue
                best_norm, score, _ = best
                e_orig = inv_emp.get(best_norm, [None])[0]
                rows.append({
                    "Company": c,
                    "Company_norm": nc,
                    "EMPRESA_best": e_orig,
                    "EMPRESA_norm": best_norm,
                    "score": float(score),
                })
            engine = "rapidfuzz"
        except Exception:
            from difflib import SequenceMatcher

            def ratio(a, b):
                return SequenceMatcher(None, a, b).ratio() * 100

            emp_choices = list(emp_norm.keys())
            for c, nc in comp_norm.items():
                best_norm = None
                best_score = -1
                for en in emp_choices:
                    sc = ratio(nc, en)
                    if sc > best_score:
                        best_score = sc
                        best_norm = en
                e_orig = inv_emp.get(best_norm, [None])[0] if best_norm is not None else None
                rows.append({
                    "Company": c,
                    "Company_norm": nc,
                    "EMPRESA_best": e_orig,
                    "EMPRESA_norm": best_norm,
                    "score": float(best_score),
                })
            engine = "difflib"

        df_fuzzy = pd.DataFrame(rows).sort_values('score', ascending=False).reset_index(drop=True)
        print(f"\nMotor de similitud: {engine}")
        print("Sugerencia: revisar coincidencias con score alto (p.ej. >= 90).")

        display(df_fuzzy.head(30))
        umbral = 90
        df_high = df_fuzzy[df_fuzzy['score'] >= umbral]
        print(f"\nCoincidencias sugeridas con score >= {umbral}: {len(df_high)}")
        display(df_high.reset_index(drop=True))

Coincidencias EXACTAS (tras normalizar): 0

Motor de similitud: rapidfuzz
Sugerencia: revisar coincidencias con score alto (p.ej. >= 90).


,Company,Company_norm,EMPRESA_best,EMPRESA_norm,score
0,OSRAM,OSRAM,CORSAM,CORSAM,72.727273
1,FEMSA,FEMSA,FADESA,FADESA,72.727273
2,SMI,SMI,BMI,BMI,66.666667
3,BIC,BIC,BAC,BAC,66.666667
4,UCB,UCB,UPC,UPC,66.666667
5,CBC,CBC,BAC,BAC,66.666667
6,Chronos,CHRONOS,CONSEP,CONSEP,61.538462
7,LACOSTE,LACOSTE,CONSEP,CONSEP,61.538462
8,BACARDI,BACARDI,BANRED,BANRED,61.538462
9,BACHOCO,BACHOCO,BAC,BAC,60.000000



Coincidencias sugeridas con score >= 90: 0


,Company,Company_norm,EMPRESA_best,EMPRESA_norm,score


### Interpretación técnica de esta validación

Si esta sección produce:

- **0 coincidencias exactas tras normalización**, y
- **0 coincidencias con score alto**,

entonces existe evidencia suficiente para afirmar que ambas fuentes no pueden integrarse de forma confiable mediante el nombre de la empresa.

Este resultado no representa una falla del código, sino un hallazgo del análisis de calidad e integración de datos.


> Nota de organización: la generación de **data cruda normalizada** (staging) y el **export** a CSV se movieron a `02_data_cleaning/01_raw_normalizado_export.ipynb` para no mezclar ingesta/profiling con cleaning.

## 7. Resultado de la validación

No se encontraron coincidencias exactas entre las empresas de ambas fuentes.

### Interpretación

Esto indica que:

- Las fuentes no están alineadas
- No es posible realizar un join directo confiable
- Existe inconsistencia en la representación de entidades

### Conclusión

Se rechaza la hipótesis de integración directa.

## 8. Próximos pasos (según evidencia)

- No forzar join `Company` ↔ `EMPRESA` por nombre: el profiling muestra que no hay coincidencias confiables.
- Preparar staging normalizado para BD/joins posteriores en `02_data_cleaning/01_raw_normalizado_export.ipynb` (export a CSV).
- Definir estrategia alternativa de integración (p.ej. catálogo unificado de empresas / matching asistido / llaves adicionales).

## 9. Universo unificado de empresas (LEADS ∪ HORAS) y verificación contra SCVS

> Objetivo: como no hay coincidencias exactas entre ambas fuentes, construimos el **universo unificado** de empresas (a partir de ambas) y evaluamos cobertura en una fuente externa (SCVS / Superintendencia de Compañías).

- Esta sección **no exporta** CSV; solo genera dataframes y métricas para el análisis.

In [7]:
# --- (A) Universo unificado (union all) de empresas normalizadas ---
def _normalize_text_for_matching_local(value) -> str:
    # Reutiliza el estilo de normalización usado en el profiling (sin reglas de negocio).
    try:
        return _normalize_for_compare(value)  # definido en la sección 4.b
    except NameError:
        # fallback defensivo si se ejecuta esta celda en aislamiento
        if value is None:
            return ''
        if isinstance(value, float) and np.isnan(value):
            return ''
        s = str(value).strip()
        if not s:
            return ''
        s = unicodedata.normalize('NFKD', s)
        s = ''.join(ch for ch in s if not unicodedata.combining(ch))
        s = s.upper()
        s = re.sub(r'[^A-Z0-9 ]+', ' ', s)
        s = re.sub(r'\s+', ' ', s).strip()
        return s

def _normalize_colkey(name) -> str:
    s = '' if name is None else str(name)
    s = unicodedata.normalize('NFKD', s)
    s = ''.join(ch for ch in s if not unicodedata.combining(ch))
    s = s.lower().strip()
    s = re.sub(r'[^a-z0-9]+', '_', s)
    s = re.sub(r'_+', '_', s).strip('_')
    return s

def _find_col_by_candidates(df: pd.DataFrame, candidates: list[str]) -> str | None:
    norm_to_actual = {_normalize_colkey(c): c for c in df.columns}
    for cand in candidates:
        k = _normalize_colkey(cand)
        if k in norm_to_actual:
            return norm_to_actual[k]
    return None

def _prep_distinct_series(df: pd.DataFrame, col: str | None) -> pd.Series:
    if not col or col not in df.columns:
        return pd.Series(dtype='string')
    s = df[col].dropna().astype('string')
    s = s.str.strip()
    s = s[s != '']
    s = s.map(_normalize_text_for_matching_local)
    s = s.replace('', pd.NA).dropna()
    # distinct dentro de cada fuente
    return s.drop_duplicates()

# Detectar columna de empresa en cada fuente (robusto a headers)
leads_company_col = _find_col_by_candidates(df_leads, ['Company', 'EMPRESA', 'Empresa', 'Razon Social', 'Razón Social', 'Nombre Empresa'])
horas_company_col = _find_col_by_candidates(df_horas, ['EMPRESA', 'Empresa', 'Company', 'CLIENTE', 'Cliente', 'Razon Social', 'Razón Social', 'Nombre Empresa'])

leads_empresas = _prep_distinct_series(df_leads, leads_company_col)
horas_empresas = _prep_distinct_series(df_horas, horas_company_col)

# union all (concat) luego de distinct por separado
empresas_union = pd.concat([horas_empresas, leads_empresas], ignore_index=True)

# Para tener un solo conjunto final, deduplicar globalmente
empresas_union = empresas_union.drop_duplicates().reset_index(drop=True)

df_empresas_union_all = pd.DataFrame({'empresa_norm': empresas_union})

print('Columna LEADS detectada:', leads_company_col)
print('Columna HORAS detectada:', horas_company_col)
print('df_empresas_union_all:', df_empresas_union_all.shape)
display(df_empresas_union_all.head(30))

Columna LEADS detectada: Company
Columna HORAS detectada: EMPRESA
df_empresas_union_all: (447, 1)


,empresa_norm
0,3DPHARMA
1,AVIS
2,ADIUM
3,ALMEXA
4,ALPER SEGUROS
5,ARAUCO
6,ASEGURADORA DEL SUR
7,ASESORIA Y CONTROL
8,AUTOSHARE
9,BAC


In [8]:
# --- (B) SCVS: cargar archivos descargados y calcular cobertura (salida limpia) ---
import warnings
warnings.filterwarnings('ignore', message='Workbook contains no default style*')

SCVS_DIR = Path('data_super_compañias')
if not SCVS_DIR.exists():
    SCVS_DIR = Path('..') / '01_data_ingestion_enrichment' / 'data_super_compañias'

# Ajustes
MATCH_THRESHOLD = 85  # prueba 80/85/90 según tolerancia
VERBOSE = False       # True si quieres ver samples/tablas

def _read_excel_sheet_names(path: Path) -> list[str]:
    xls = pd.ExcelFile(path)
    return list(xls.sheet_names)

def _read_csv_best_effort(path: Path) -> pd.DataFrame | None:
    encodings = ['utf-8-sig', 'utf-8', 'latin-1', 'cp1252']
    seps = [',', ';', '\t', '|']
    for enc in encodings:
        for sep in seps:
            try:
                df = pd.read_csv(path, encoding=enc, sep=sep)
                if df is not None and not df.empty and df.shape[1] >= 2:
                    return df
            except Exception:
                continue
    for enc in encodings:
        try:
            df = pd.read_csv(path, encoding=enc, sep=None, engine='python')
            if df is not None and not df.empty and df.shape[1] >= 2:
                return df
        except Exception:
            continue
    return None

def _coerce_ruc(value):
    if value is None:
        return pd.NA
    if isinstance(value, float) and np.isnan(value):
        return pd.NA
    s = str(value).strip()
    if not s:
        return pd.NA
    s = re.sub(r'\D+', '', s)
    return s if s else pd.NA

def _is_ruc_like(series: pd.Series) -> float:
    s = series.astype('string').fillna('')
    s = s.str.replace(r'\.0$', '', regex=True)
    d = s.str.replace(r'\D+', '', regex=True)
    ok = d.str.len().between(10, 13)
    return float(ok.mean())

def _is_name_like(series: pd.Series) -> float:
    s = series.astype('string').fillna('')
    s = s.map(lambda x: x.strip())
    s = s[s != '']
    if len(s) == 0:
        return 0.0
    has_letters = s.str.contains(r'[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]', regex=True)
    avg_len = s.str.len().mean()
    return float(has_letters.mean()) * float(min(avg_len / 30.0, 1.0))

def _extract_scvs_from_df(df: pd.DataFrame, fuente_archivo: str, fuente_hoja: str | None = None) -> pd.DataFrame | None:
    if df is None or df.empty:
        return None

    name_candidates = [
        'razon social', 'razón social', 'razon_social', 'nombre', 'nombre compania', 'nombre compañía',
        'denominacion', 'denominación', 'compania', 'compañia', 'institucion', 'institución', 'empresa', 'entidad'
    ]
    ruc_candidates = [
        'ruc', 'identificacion', 'identificación', 'identificacion tributaria', 'identificación tributaria',
        'numero identificacion', 'número identificación', 'numero_de_identificacion', 'nro_identificacion',
        'documento', 'numero_documento', 'nro_documento'
    ]
    employees_candidates = ['empleados', 'n_empleados', 'numero empleados', 'número empleados', 'personal']

    col_name = _find_col_by_candidates(df, name_candidates)
    col_ruc = _find_col_by_candidates(df, ruc_candidates)
    col_emp = _find_col_by_candidates(df, employees_candidates)

    if col_name is not None:
        tmp = pd.DataFrame({
            'nombre_oficial_raw': df[col_name],
            'ruc': (df[col_ruc] if col_ruc is not None else pd.NA),
            'n_empleados': (df[col_emp] if col_emp is not None else pd.NA),
        })
    else:
        ruc_col = None
        best_ruc = 0.0
        for c in df.columns:
            score = _is_ruc_like(df[c])
            if score > best_ruc:
                best_ruc = score
                ruc_col = c
        name_col = None
        best_name = 0.0
        for c in df.columns:
            if c == ruc_col:
                continue
            score = _is_name_like(df[c])
            if score > best_name:
                best_name = score
                name_col = c
        if name_col is None and ruc_col is None:
            return None
        tmp = pd.DataFrame({
            'nombre_oficial_raw': (df[name_col] if name_col is not None else pd.NA),
            'ruc': (df[ruc_col] if ruc_col is not None else pd.NA),
            'n_empleados': pd.NA,
        })

    tmp['ruc'] = tmp['ruc'].map(_coerce_ruc)
    tmp['nombre_oficial_raw'] = tmp['nombre_oficial_raw'].astype('string')
    tmp['nombre_oficial_raw'] = tmp['nombre_oficial_raw'].fillna('').map(lambda x: x.strip())
    tmp = tmp[tmp['nombre_oficial_raw'] != '']
    tmp['nombre_oficial_norm'] = tmp['nombre_oficial_raw'].map(_normalize_text_for_matching_local)
    tmp = tmp[tmp['nombre_oficial_norm'].astype('string').str.strip() != '']
    tmp['fuente_archivo'] = fuente_archivo
    tmp['fuente_hoja'] = '' if fuente_hoja is None else str(fuente_hoja)
    return tmp.reset_index(drop=True)

def _extract_scvs_from_excel_sheet(path: Path, sheet_name: str) -> pd.DataFrame | None:
    try:
        df = pd.read_excel(path, sheet_name=sheet_name)
        part = _extract_scvs_from_df(df, fuente_archivo=path.name, fuente_hoja=sheet_name)
        if part is not None and len(part):
            return part
    except Exception:
        pass
    try:
        raw = pd.read_excel(path, sheet_name=sheet_name, header=None)
        part = _extract_scvs_from_df(raw, fuente_archivo=path.name, fuente_hoja=sheet_name)
        return part
    except Exception:
        return None

# 1) Cargar SCVS (xlsx + csv)
scvs_excels = sorted([p for p in SCVS_DIR.glob('*.xlsx') if p.is_file()])
scvs_csvs = sorted([p for p in SCVS_DIR.glob('*.csv') if p.is_file()])

scvs_parts: list[pd.DataFrame] = []
for path in scvs_excels:
    for sheet_name in _read_excel_sheet_names(path):
        part = _extract_scvs_from_excel_sheet(path, sheet_name)
        if part is not None and len(part):
            scvs_parts.append(part)

for path in scvs_csvs:
    df = _read_csv_best_effort(path)
    part = _extract_scvs_from_df(df, fuente_archivo=path.name, fuente_hoja=None)
    if part is not None and len(part):
        scvs_parts.append(part)

if not scvs_parts:
    raise RuntimeError('No se pudo extraer ninguna tabla SCVS desde los archivos en la carpeta.')

df_scvs_master = pd.concat(scvs_parts, ignore_index=True)
df_scvs_master = df_scvs_master.dropna(subset=['nombre_oficial_norm'])
df_scvs_master = df_scvs_master.drop_duplicates(subset=['nombre_oficial_norm', 'ruc'])

# 2) Preparar universo
universe = df_empresas_union_all['empresa_norm'].astype('string')
universe = universe.fillna('').map(lambda x: x.strip())
universe = universe[universe != ''].drop_duplicates().reset_index(drop=True)

choices = df_scvs_master['nombre_oficial_norm'].astype('string').tolist()
exact_set = set(choices)
n_exact = int(universe.isin(exact_set).sum())

# 3) Matching fuzzy (auto-instala rapidfuzz si falta)
engine = None
extract_one = None

try:
    from rapidfuzz import process as rf_process, fuzz as rf_fuzz
    engine = 'rapidfuzz'
    def extract_one(q: str):
        return rf_process.extractOne(q, choices, scorer=rf_fuzz.token_set_ratio)
except Exception:
    try:
        import sys, subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'rapidfuzz'])
        from rapidfuzz import process as rf_process, fuzz as rf_fuzz
        engine = 'rapidfuzz'
        def extract_one(q: str):
            return rf_process.extractOne(q, choices, scorer=rf_fuzz.token_set_ratio)
    except Exception:
        engine = None
        extract_one = None

if extract_one is None:
    try:
        from thefuzz import process as fz_process, fuzz as fz_fuzz
        engine = 'thefuzz'
        def extract_one(q: str):
            return fz_process.extractOne(q, choices, scorer=fz_fuzz.token_set_ratio)
    except Exception:
        from difflib import SequenceMatcher
        engine = 'difflib'
        def ratio(a: str, b: str) -> float:
            return SequenceMatcher(None, a, b).ratio() * 100
        def extract_one(q: str):
            best = None
            best_score = -1.0
            for cand in choices:
                sc = ratio(q, cand)
                if sc > best_score:
                    best = cand
                    best_score = sc
            return (best, best_score)

# Índice para recuperar datos SCVS (incluye nombre raw + RUC)
scvs_by_norm = (df_scvs_master.sort_values(['ruc']).drop_duplicates(subset=['nombre_oficial_norm'], keep='first')
                .set_index('nombre_oficial_norm'))

rows_match = []
for q in universe.tolist():
    res = extract_one(q)
    if res is None:
        rows_match.append({
            'empresa_norm': q,
            'scvs_nombre_norm': pd.NA,
            'scvs_nombre_raw': pd.NA,
            'status': 'no_match',
            'score': 0.0,
            'ruc': pd.NA,
        })
        continue
    if engine in ('rapidfuzz', 'thefuzz'):
        best_norm, score, *_ = res
        score = float(score)
    else:
        best_norm, score = res
        score = float(score)
    status = 'matched' if score >= MATCH_THRESHOLD else 'below_threshold'
    rec = scvs_by_norm.loc[best_norm] if best_norm in scvs_by_norm.index else None
    rows_match.append({
        'empresa_norm': q,
        'scvs_nombre_norm': best_norm,
        'scvs_nombre_raw': (rec['nombre_oficial_raw'] if rec is not None else pd.NA),
        'score': score,
        'status': status,
        'ruc': (rec['ruc'] if rec is not None else pd.NA),
    })

df_empresas_scvs_match = pd.DataFrame(rows_match)

n_total = len(df_empresas_scvs_match)
n_ok = int((df_empresas_scvs_match['status'] == 'matched').sum())
n_low = int((df_empresas_scvs_match['status'] == 'below_threshold').sum())

ruc_digits = df_empresas_scvs_match['ruc'].astype('string').fillna('')
ruc_len_13 = ruc_digits.str.len().eq(13)
n_ruc_ok = int(((df_empresas_scvs_match['status'] == 'matched') & (df_empresas_scvs_match['ruc'].notna())).sum())
n_ruc13_ok = int(((df_empresas_scvs_match['status'] == 'matched') & ruc_len_13).sum())

print('[SCVS] Resumen')
print('- universo_empresas:', n_total)
print('- scvs_registros_cargados:', int(len(df_scvs_master)))
print('- exact_matches_norm:', n_exact)
print(f'- fuzzy_matches_>=_{MATCH_THRESHOLD}:', n_ok)
print(f'- fuzzy_matches_<_ {MATCH_THRESHOLD}:', n_low)
print(f'- matched_con_RUC_no_nulo (>= {MATCH_THRESHOLD}):', n_ruc_ok)
print(f'- matched_con_RUC_13_digitos (>= {MATCH_THRESHOLD}):', n_ruc13_ok)
print('- motor_fuzzy:', engine)

if VERBOSE:
    display(df_scvs_master.head(5))
    display(df_empresas_scvs_match.sort_values('score', ascending=False).head(20))

[SCVS] Resumen
- universo_empresas: 447
- scvs_registros_cargados: 219439
- exact_matches_norm: 1
- fuzzy_matches_>=_85: 146
- fuzzy_matches_<_ 85: 301
- matched_con_RUC_no_nulo (>= 85): 145
- matched_con_RUC_13_digitos (>= 85): 145
- motor_fuzzy: rapidfuzz


In [9]:
# --- (B2) Overrides manuales (instituciones públicas / bancos) ---
manual_mapping = {
    "CORPORACION GPF": {"ruc": "1790403755001", "nombre_oficial": "ECONOFARM S.A."},
    "EEQ": {"ruc": "1768153530001", "nombre_oficial": "EMPRESA ELECTRICA QUITO S.A."},
    "ESPE": {"ruc": "1768133420001", "nombre_oficial": "UNIV. DE LAS FUERZAS ARMADAS ESPE"},
    "MINISTERIO SALUD PUBLICA": {"ruc": "1768040760001", "nombre_oficial": "MINISTERIO DE SALUD PUBLICA"},
    "MUNICIPIO DE QUITO": {"ruc": "1760001550001", "nombre_oficial": "MUNICIPIO DEL DISTRITO METROPOLITANO"},
    "CUERPO INGENIEROS EJERCITO": {"ruc": "1768001600001", "nombre_oficial": "CUERPO DE INGENIEROS DEL EJERCITO"},
    "HIDROPAUTE": {"ruc": "1768152800001", "nombre_oficial": "CELEC EP"},
    "PRESIDENCIA": {"ruc": "1768000200001", "nombre_oficial": "PRESIDENCIA DE LA REPUBLICA"},
    "BANCO DEL AUSTRO": {"ruc": "0190035042001", "nombre_oficial": "BANCO DEL AUSTRO S.A."},
    "DINERS": {"ruc": "1790151706001", "nombre_oficial": "DINERS CLUB DEL ECUADOR S.A."},
    "BANRED": {"ruc": "1791334758001", "nombre_oficial": "BANRED S.A."},
    "EL TELEGRAFO": {"ruc": "1768156200001", "nombre_oficial": "EMPRESA PUBLICA EL TELEGRAFO EP"},
    "UTN": {"ruc": "1060002770001", "nombre_oficial": "UNIVERSIDAD TECNICA DEL NORTE"},
    "UTPL": {"ruc": "1190068729001", "nombre_oficial": "UNIVERSIDAD TECNICA PARTICULAR DE LOJA"},
    "PUCE": {"ruc": "1790105304001", "nombre_oficial": "PONTIFICIA UNIVERSIDAD CATOLICA DEL ECUADOR"},
}

if 'df_empresas_scvs_match' not in globals():
    raise RuntimeError('No existe df_empresas_scvs_match. Ejecuta primero la celda de matching SCVS.')

# Crear/llenar columna de verificación (qué nombre se usará como "match" final)
if 'nombre_oficial_match' not in df_empresas_scvs_match.columns:
    if 'scvs_nombre_raw' in df_empresas_scvs_match.columns:
        df_empresas_scvs_match['nombre_oficial_match'] = df_empresas_scvs_match['scvs_nombre_raw']
    else:
        df_empresas_scvs_match['nombre_oficial_match'] = pd.NA

empresa_ser = df_empresas_scvs_match['empresa_norm'].astype('string').fillna('').map(lambda x: x.strip())

n_applied = 0
missing = []
for empresa_key, payload in manual_mapping.items():
    key = str(empresa_key).strip()
    mask = empresa_ser.eq(key)
    if not mask.any():
        missing.append(key)
        continue
    df_empresas_scvs_match.loc[mask, 'ruc'] = payload.get('ruc', pd.NA)
    df_empresas_scvs_match.loc[mask, 'nombre_oficial_match'] = payload.get('nombre_oficial', pd.NA)
    df_empresas_scvs_match.loc[mask, 'status'] = 'manual_override'
    df_empresas_scvs_match.loc[mask, 'score'] = 100.0
    n_applied += int(mask.sum())

print(f"[MANUAL] overrides aplicados: {n_applied} de {len(manual_mapping)} claves")
if missing:
    print(f"[MANUAL] claves no encontradas en empresa_norm: {len(missing)}")

[MANUAL] overrides aplicados: 15 de 15 claves


In [10]:
import pandas as pd

# 1. Definir los datos generados por la IA
data = [
    {"empresa_norm": "3DPHARMA", "ai_estimated_industry": "Healthcare & Pharmaceuticals", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ADIUM", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ALMEXA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ALPER SEGUROS", "ai_estimated_industry": "Insurance", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ARAUCO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "AUTOSHARE", "ai_estimated_industry": "Automotive", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "BANCO FINANCIERO", "ai_estimated_industry": "Financial Services", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "BEKAERT", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "BEMIS", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "BITSO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "BLOOMIN BRANDS", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "BOSTON BSCI", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "CETI VEHICULOS", "ai_estimated_industry": "Automotive", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "CHARGERLOGISTICS", "ai_estimated_industry": "Logistics & Transportation", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "CLINTON FOUNDATION", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "COLSUBSIDIO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "COMESTIBLES INTEGRALES", "ai_estimated_industry": "Food & Beverage", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "DEPRATI", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ELOSERVICOS", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "FARMAPIEL", "ai_estimated_industry": "Healthcare & Pharmaceuticals", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "HEINSOHN TECH", "ai_estimated_industry": "Technology & Telecom", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "INDUGLOB", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "KOMATSU", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "MAYFLOWER BUFFALO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "MILLICOM TELEFONICA PA NI", "ai_estimated_industry": "Technology & Telecom", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "MODEC", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "MODEC GUYANA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "NOVA ECUADOR", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Ecuador"},
    {"empresa_norm": "OLVA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "PUCE", "ai_estimated_industry": "Education", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Ecuador"},
    {"empresa_norm": "PRIOX", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "QUINTO ANDAR", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "REDCAPITAL", "ai_estimated_industry": "Financial Services", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "SEPS", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "SUPERTEL", "ai_estimated_industry": "Technology & Telecom", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "TCG GROUP", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "TELEFONICA CR", "ai_estimated_industry": "Technology & Telecom", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "TELEFONICA EC", "ai_estimated_industry": "Technology & Telecom", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Ecuador"},
    {"empresa_norm": "TELMEX", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "TELNA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "TERNIUM", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "UTN", "ai_estimated_industry": "Education", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Ecuador"},
    {"empresa_norm": "UTPL", "ai_estimated_industry": "Education", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Ecuador"},
    {"empresa_norm": "UNIVERSIDAD JAVERIANA CALI", "ai_estimated_industry": "Education", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "VEOLIA ARGENTINA", "ai_estimated_industry": "Energy & Utilities", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Argentina"},
    {"empresa_norm": "VEOLIA BRASIL", "ai_estimated_industry": "Energy & Utilities", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Brazil"},
    {"empresa_norm": "VEOLIA CHILE", "ai_estimated_industry": "Energy & Utilities", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Chile"},
    {"empresa_norm": "VEOLIA CHINA", "ai_estimated_industry": "Energy & Utilities", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "China"},
    {"empresa_norm": "VEOLIA COLOMBIA", "ai_estimated_industry": "Energy & Utilities", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Colombia"},
    {"empresa_norm": "VEOLIA JAPON", "ai_estimated_industry": "Energy & Utilities", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Japan"},
    {"empresa_norm": "VEOLIA LATAM", "ai_estimated_industry": "Energy & Utilities", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Multinational (LatAm/Global)"},
    {"empresa_norm": "VEOLIA MEXICO", "ai_estimated_industry": "Energy & Utilities", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "VEOLIA PERU", "ai_estimated_industry": "Energy & Utilities", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Peru"},
    {"empresa_norm": "WEBMOTORS", "ai_estimated_industry": "Automotive", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GAMEPLANET S A DE C V", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "ALCIONE MX", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "ASTRAZENECA", "ai_estimated_industry": "Healthcare & Pharmaceuticals", "ai_estimated_size": "Enterprise (5000+)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "FARMACIAS DEL AHORRO", "ai_estimated_industry": "Healthcare & Pharmaceuticals", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "DHL EXPRESS MEXICO", "ai_estimated_industry": "Logistics & Transportation", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "ARCA CONTINENTAL", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "PRIMERO SEGUROS", "ai_estimated_industry": "Insurance", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "BACARDI", "ai_estimated_industry": "Food & Beverage", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GEFCO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GNP SEGUROS", "ai_estimated_industry": "Insurance", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "COPPEL", "ai_estimated_industry": "Retail", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "CARNOT LABORATORIOS", "ai_estimated_industry": "Healthcare & Pharmaceuticals", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "KLASSCO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "MARSH", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "FARMACIAS BENAVIDES", "ai_estimated_industry": "Healthcare & Pharmaceuticals", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "FIDUCIARIA DE OCCIDENTE", "ai_estimated_industry": "Financial Services", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "PELICULAS PLASTICAS", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "CREEL GARCIA CUELLAR AIZA Y ENRIQUEZ", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "INGREDION INCORPORATED", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "MAVER", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "XIGNUX", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO ALEN", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "KOF", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO COPPEL", "ai_estimated_industry": "Retail", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "GRUPO SALINAS", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "EL PALACIO DE HIERRO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ATRADIUS SEGUROS DE CREDITO", "ai_estimated_industry": "Insurance", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ACCO BRANDS", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO SID", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "WIGOOBOX", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "MCM TELCOM", "ai_estimated_industry": "Technology & Telecom", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "JUGUETRON", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "HOME INTERIORS", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "STANHOME", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ORGANIZACION SORIANA", "ai_estimated_industry": "Retail", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "JEMAFLEX DE MEXICO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "ALLIANZ MEXICO", "ai_estimated_industry": "Insurance", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "LECHE ANDINA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRISI HNOS", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO HOTELERO BRISAS", "ai_estimated_industry": "Hospitality & Entertainment", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ELI LILLY Y CIA DE MEXICO S A DE CV", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "CUAUHTEMOC MOCTEZUMA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "HEINEKEN MEXICO", "ai_estimated_industry": "Food & Beverage", "ai_estimated_size": "Enterprise (5000+)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "SUMA GLOBAL PACK", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Multinational (LatAm/Global)"},
    {"empresa_norm": "ANHEUSER BUSCH INBEV", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "UNIVERSIDAD ANAHUAC", "ai_estimated_industry": "Education", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "FERRERO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "CONAGRA FOODS", "ai_estimated_industry": "Food & Beverage", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "SANDOZ", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "TECNOLOGICO DE MONTERREY", "ai_estimated_industry": "Education", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GLOBAL HITSS", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Multinational (LatAm/Global)"},
    {"empresa_norm": "GRUPO MINSA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "CORONA ATIZAPAN", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "PEPSICO", "ai_estimated_industry": "Food & Beverage", "ai_estimated_size": "Enterprise (5000+)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "AMBROSIA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "MOKSHA8", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "WHIRLPOOL", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO VASCONIA SAB", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "CLAYTON DE MEXICO S A DE C V", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "CLG TRANSPORTES", "ai_estimated_industry": "Logistics & Transportation", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO MYM", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "SEGUROS DE MONTERREY NEW YORK LIFE", "ai_estimated_industry": "Insurance", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "VIVE SEGUROS", "ai_estimated_industry": "Insurance", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "METLIFE MEXICO", "ai_estimated_industry": "Insurance", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "ELICA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO RON DIPLOMATICO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "THE HOME DEPOT MEXICO", "ai_estimated_industry": "Retail", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "URBI", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "NOVAVENTA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "TRANE", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "PAPELES CORRUGADOS", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO BIMBO", "ai_estimated_industry": "Food & Beverage", "ai_estimated_size": "Enterprise (5000+)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "FABRICAS DE CALZADO ANDREA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "MATTEL LATIN AMERICA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "NACIONAL MONTE DE PIEDAD", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "IFF", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "STAND DELIVER GROUP", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ELECTRO JAPONESA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "CONFITECA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GENOMMA LAB", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO HIDROSINA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "DENTEGRA SEGUROS DENTALES", "ai_estimated_industry": "Insurance", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "KIMBERLY CLARK", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ONTEX", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "TESALIA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "STEVE MADDEN", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "PRODIGY", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "MC CORMICK PESA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ITALMEX PHARMA", "ai_estimated_industry": "Healthcare & Pharmaceuticals", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "WALMART", "ai_estimated_industry": "Retail", "ai_estimated_size": "Enterprise (5000+)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "NATURESWEET", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO ZUCARMEX", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "POZUELO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "LIVERPOOL", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "CUETARA COOKIES", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUNENTHAL", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "EGADE BUSINESS SCHOOL DEL TECNOLOLGICO DE MONTERREY", "ai_estimated_industry": "Education", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "EUROFARMA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "MEDIX", "ai_estimated_industry": "Healthcare & Pharmaceuticals", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "KCC", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "SEAGATE", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ESACERO S A", "ai_estimated_industry": "Construction & Materials", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO CHEDRAUI", "ai_estimated_industry": "Retail", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "CASA CUERVO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "R S HUGHES", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "MERCK", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "THE MIDDLEBY CORPORATION", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "FORZASTEEL", "ai_estimated_industry": "Construction & Materials", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO EDUCARE", "ai_estimated_industry": "Education", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "UNIVERSIDAD PANAMERICANO", "ai_estimated_industry": "Education", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "KIMBERLY CLARK DE MEXICO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "ABBOTT", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Enterprise (5000+)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "AIG", "ai_estimated_industry": "Insurance", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GIGANTE", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "HILTI", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO MODELO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "DAWN FOODS", "ai_estimated_industry": "Food & Beverage", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "COLCHONES WENDY S A", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ITESO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO CARSO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "LACOSTE", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ASAP TESTING", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "SEGUROS MONTERREY NEW YORK LIFE", "ai_estimated_industry": "Insurance", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "COCA COLA", "ai_estimated_industry": "Food & Beverage", "ai_estimated_size": "Enterprise (5000+)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GLP", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ROYAL RESORTS", "ai_estimated_industry": "Hospitality & Entertainment", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "FELLOWES BRANDS", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "PISA FARMACEUTICA", "ai_estimated_industry": "Healthcare & Pharmaceuticals", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "CBC", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "FIBRA INN", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "KELLOGG COMPANY", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "LECHE SAN MARCOS", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "NOVATHINKA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO POSADAS DE MEXICO S A DE C V", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "PROVEEDORES DE INGENIERIA DE ALIMENTOS", "ai_estimated_industry": "Food & Beverage", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "LLANO DE LA TORRE", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO BIOSSMANN", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "PRODES PROMOTORA DE DESHIDRATADOS", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "KELLOG", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "7 ELEVEN MEXICO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "AZUCAR GRUPO SAENZ", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "AFP GENESIS", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ROSHFRANS MX", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "GRUPO MEXICANO DE SEGUROS", "ai_estimated_industry": "Insurance", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "NEOLPHARMA S A DE C V", "ai_estimated_industry": "Healthcare & Pharmaceuticals", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "GRUPO SOMAR", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "PHILLIPS", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ALPEZZI CHOCOLATE", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "KUA MEX FOODS", "ai_estimated_industry": "Food & Beverage", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "OFFICE DEPOT", "ai_estimated_industry": "Retail", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GREENBERG TRAURIG", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO CHOCOLATE IBARRA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO ADOROTE", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO GALERIA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "INDUATENAS", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO CALIFA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ACH FOOD COMPANIES INC", "ai_estimated_industry": "Food & Beverage", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO PIASA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO ROTOPLAS", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "APOTEX", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO TMM", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "LA ANITA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "CHARDON MEXICO ELECTRIC S A DE C V", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "GRUPO AMPM", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "CHUBB", "ai_estimated_industry": "Insurance", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "LABORATORIOS SOPHIA", "ai_estimated_industry": "Healthcare & Pharmaceuticals", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "LEGO GROUP", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ARCOR", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "SPECTRUM BRANDS INC", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO INDUSTRIAL SALTILLO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "AVERY DENNISON RBIS", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "SAMSUNG ELECTRONICS MEXICO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Enterprise (5000+)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "ISASTUR", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "MILLICOM", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "MTWA SOLUCIONES INTEGRALES", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO BITUAJ", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "BOEHRINGER INGELHEIM MEXICO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "GBT BIOTOSCANA GROUP", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "FRAMACIAS BENAVIDES", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "PARAUCO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "INCALPACA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "PINTURAS BERELS A DE C V", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "MUMUSO MEXICO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "GRUPO ALTEX", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "PPG", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "QUALTIA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "CMR MEXICO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "GRIFFITH FOODS", "ai_estimated_industry": "Food & Beverage", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "HDI SEGUROS M AXICO", "ai_estimated_industry": "Insurance", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "INOVA MEXICO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "UCB", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ALVAREZ BARBA S A", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Small (1-50)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "BMS AT REDWOOD CITY", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "SUNOPTA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "SYNTHON", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "NEWELL BRANDS", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "NOVARTIS", "ai_estimated_industry": "Healthcare & Pharmaceuticals", "ai_estimated_size": "Enterprise (5000+)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "JUGOS DEL VALLE", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ESSITY", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "LOCATION WORLD", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "FANASA FARMACOS NACIONALES", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "HDI SEGUROS MEXICO", "ai_estimated_industry": "Insurance", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "HI CONE MEXICO ENVASES MULTIPAC", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "CEMEX", "ai_estimated_industry": "Construction & Materials", "ai_estimated_size": "Enterprise (5000+)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "GRUPO INDUSTRIAL MONCLOVA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "TAJIN", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "CID GRUPO KNOBLOCH", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "COLGATE PALMOLIVE", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "FEMSA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "BAXTER", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "MAGNA STEYR", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ONAPIS", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO CUPRUM", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ECONOFARM S A CORPORACION GPF", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "IGSA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "NADER HAYAUX GOEBEL", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "NOVAMEX", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "HOTELES CITY EXPRESS", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO INDUSTRIALMONCLOVA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "FREELANCE SELF EMPLOYED", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "BPL BIO PRODUCTS LABORATORY", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "NADRO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "MODA HOLDING", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO TRIMEX S A", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "FARMACIAS DE SIMILARES", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO AXO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ALIMENTOSALCONSUMIDOR", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "LANDSTEINER SCIENTIFIC", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "CFB", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GUNENTHAL", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ULTRA LABORATAORIOS", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "LITCORP", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "CHIESI GROUP", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "AUTO TODO MEXICANA S A DE C V", "ai_estimated_industry": "Automotive", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "METLIFE", "ai_estimated_industry": "Insurance", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "SAMSONITE", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "DANONE", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO FAMILIA S A", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GROUPE SEB", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ESTAFETA MEXICANA", "ai_estimated_industry": "Logistics & Transportation", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "ASIAUTO S A KIA MOTORS ECUADOR", "ai_estimated_industry": "Automotive", "ai_estimated_size": "Small (1-50)", "ai_estimated_country": "Ecuador"},
    {"empresa_norm": "CALAHUA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ENERGIZER", "ai_estimated_industry": "Energy & Utilities", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GEPP", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "CENTRO DE ENTRENAMIENTO DE TERAPIA ENDOVASCULAR", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO LEVBETH", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "V TECHNOLOGIES INVESTING V", "ai_estimated_industry": "Technology & Telecom", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO DEINSAT", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ACTINVER", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "COMEX", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "LIOMONT LABORATORIES", "ai_estimated_industry": "Healthcare & Pharmaceuticals", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "DIRECTV", "ai_estimated_industry": "Technology & Telecom", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "BIIS LOGISTICS", "ai_estimated_industry": "Logistics & Transportation", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ABBVIE", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "TALMEX PHARMA", "ai_estimated_industry": "Healthcare & Pharmaceuticals", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "BECLE SAB", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "JANSSSEN CILAG", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "JUGUETIBICI", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "PRODUCTOS MAVER", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "FABRICA DE HARINAS ELIZONDO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "PINSA COMERCIAL", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "PROVEEDORES DE INGENIERIA ALIMENTARIA S A DE C V", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "FGX INTERNACIONAL", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Multinational (LatAm/Global)"},
    {"empresa_norm": "CHINOIN PRODUCTOS FARMACEUTICOS S A DE C V", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUPO GAYOSSO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "CORPORACION MOCTEZUMA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Large (501-5000)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ROCHE MEXICO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "FORZAC CONCRETOS PREMIER", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "CXO FORUM", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "CAMARA NAC DE LA INDUSTRIA DE RESTAURANTES Y ALIMENTOS", "ai_estimated_industry": "Food & Beverage", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "STE SOLUCIONES TECNOLOGICAS", "ai_estimated_industry": "Technology & Telecom", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "TOKIO MARINE CIA DE SGUROS", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "LEONALI S DE R L DE C V", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "TELEFONICA", "ai_estimated_industry": "Technology & Telecom", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "PRODUCTOS DE CONSUMO Z SA DE CV", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "PROESSA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "GRUMA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "RENT A CENTER", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "AKROS", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "TALMA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "OSRAM", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "MIFLINK", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "CLARO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ADAMANTINE", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "AVON", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "FRESENIUS MEDICAL CARE", "ai_estimated_industry": "Healthcare & Pharmaceuticals", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "EPSON MEXICO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "BACCHUS CONSULTING GROUP", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "BACCHUS WINE AMP SPIRITS SHOP", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "BACHOCO", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ATTEBURY GRAIN LLC", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "SALJAMEX", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "FIRSTCALL CSS", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "INTER CON SECURITY SYSTEMS INC", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "CITROFRUT", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "LABORATORIO DE CONTROL ARJ", "ai_estimated_industry": "Healthcare & Pharmaceuticals", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "THERAPEDIC MEXICO", "ai_estimated_industry": "Healthcare & Pharmaceuticals", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Mexico"},
    {"empresa_norm": "CHRONOS", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "ICONN", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "MOKSHA9", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "SYM", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "OCEAN BY H10 HOTELS", "ai_estimated_industry": "Hospitality & Entertainment", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "MEDIPASSRED", "ai_estimated_industry": "Healthcare & Pharmaceuticals", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "WRCA", "ai_estimated_industry": "Manufacturing & Conglomerates", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"},
    {"empresa_norm": "PETROLI", "ai_estimated_industry": "Energy & Utilities", "ai_estimated_size": "Medium (51-500)", "ai_estimated_country": "Unknown/Multinational"}
]

df_enrichment = pd.DataFrame(data)

# 2. Hacer un merge (Left Join) con tu DataFrame original df_empresas_scvs_match
# Suponiendo que tu dataframe original se llama df_empresas_scvs_match

# Hacemos el cruce usando 'empresa_norm' como llave
df_final_cluster = pd.merge(
    df_empresas_scvs_match, 
    df_enrichment, 
    on='empresa_norm', 
    how='left'
 )

# Ahora df_final_cluster tiene las nuevas columnas 'ai_estimated_industry', 'ai_estimated_size', 'ai_estimated_country'
print(df_final_cluster.head())

    empresa_norm scvs_nombre_norm   scvs_nombre_raw       score           status            ruc nombre_oficial_match          ai_estimated_industry ai_estimated_size   ai_estimated_country
0       3DPHARMA  593PHARMA S A S  593PHARMA S.A.S.   66.666667  below_threshold  1891811969001     593PHARMA S.A.S.   Healthcare & Pharmaceuticals   Medium (51-500)  Unknown/Multinational
1           AVIS    AVIS CIA LTDA    AVIS CIA.LTDA.  100.000000          matched  1793119549001       AVIS CIA.LTDA.                            NaN               NaN                    NaN
2          ADIUM       RODIUM S A       RODIUM S.A.   66.666667  below_threshold  0993265837001          RODIUM S.A.  Manufacturing & Conglomerates   Medium (51-500)  Unknown/Multinational
3         ALMEXA       MEXA S A S       MEXA S.A.S.   71.428571  below_threshold  1793214892001          MEXA S.A.S.  Manufacturing & Conglomerates   Medium (51-500)  Unknown/Multinational
4  ALPER SEGUROS   LEOSEGUROS S A   LEOSEGUROS S.A.   7

In [11]:
cols = [
    'empresa_norm',
    'ai_estimated_industry',
    'ai_estimated_size',
    'ai_estimated_country',
    'ruc',
    'status',
    'score',
    'nombre_oficial_match',
]

mask_uni = df_final_cluster['empresa_norm'].isin(['UTN', 'UTPL', 'PUCE'])
print(df_final_cluster.loc[mask_uni, cols].sort_values('empresa_norm').to_string(index=False))

empresa_norm ai_estimated_industry ai_estimated_size ai_estimated_country           ruc          status  score                        nombre_oficial_match
        PUCE             Education  Large (501-5000)              Ecuador 1790105304001 manual_override  100.0 PONTIFICIA UNIVERSIDAD CATOLICA DEL ECUADOR
         UTN             Education  Large (501-5000)              Ecuador 1060002770001 manual_override  100.0               UNIVERSIDAD TECNICA DEL NORTE
        UTPL             Education  Large (501-5000)              Ecuador 1190068729001 manual_override  100.0      UNIVERSIDAD TECNICA PARTICULAR DE LOJA


In [12]:
# --- (C) Export a CSV del resultado SCVS (descarga local) ---
from pathlib import Path

# Este CSV contiene el mapeo: empresa_norm -> score/status -> ruc (si se pudo extraer)
if 'df_empresas_scvs_match' not in globals():
    raise RuntimeError('No existe df_empresas_scvs_match. Ejecuta primero la celda anterior (SCVS).')

def _clean_df_for_csv(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]
    for col in out.columns:
        if pd.api.types.is_string_dtype(out[col]) or pd.api.types.is_object_dtype(out[col]):
            s = out[col].astype('string')
            s = s.str.replace('\u00a0', ' ', regex=False).str.strip()
            out[col] = s
    return out

df_export = _clean_df_for_csv(df_empresas_scvs_match)

out_dir = Path('outputs')
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / f"scvs_matches_threshold_{MATCH_THRESHOLD}.csv"
df_export.to_csv(out_path, index=False, encoding='utf-8-sig')

print('[EXPORT] Archivo CSV generado:', str(out_path.resolve()))
print('[EXPORT] Filas/Columnas:', df_export.shape)

[EXPORT] Archivo CSV generado: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\01_data_ingestion_enrichment\outputs\scvs_matches_threshold_85.csv
[EXPORT] Filas/Columnas: (447, 7)


In [13]:
# (C.1) Limpieza del CSV exportado (quita padding de espacios)
from pathlib import Path
import pandas as pd

raw_path = Path('outputs') / f"scvs_matches_threshold_{MATCH_THRESHOLD}.csv"
clean_path = raw_path.with_name(raw_path.stem + '_clean' + raw_path.suffix)

if raw_path.exists():
    df_tmp = pd.read_csv(raw_path, sep=',', engine='python', dtype=str)
    df_tmp.columns = [str(c).strip() for c in df_tmp.columns]
    df_tmp = df_tmp.apply(lambda s: s.astype('string').str.replace('\u00a0', ' ', regex=False).str.strip())
    df_tmp.to_csv(clean_path, index=False, encoding='utf-8-sig')
    print('[CLEAN] OK ->', str(clean_path.resolve()))
    print('[CLEAN] Filas/Columnas:', df_tmp.shape)
else:
    print('[CLEAN] No existe:', str(raw_path.resolve()))

[CLEAN] OK -> E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\01_data_ingestion_enrichment\outputs\scvs_matches_threshold_85_clean.csv
[CLEAN] Filas/Columnas: (447, 7)


In [14]:
# (C.2) Verificación rápida del header: raw vs clean
from pathlib import Path

raw_path = Path('outputs') / f"scvs_matches_threshold_{MATCH_THRESHOLD}.csv"
clean_path = raw_path.with_name(raw_path.stem + '_clean' + raw_path.suffix)

for p in [raw_path, clean_path]:
    if not p.exists():
        print('[CHECK] missing:', str(p.resolve()))
        continue
    with p.open('r', encoding='utf-8-sig', errors='replace') as f:
        line = f.readline().rstrip('\n')
    print('[CHECK]', p.name)
    print('  repr:', repr(line[:120]))
    print('  len :', len(line))

[CHECK] scvs_matches_threshold_85.csv
  repr: 'empresa_norm,scvs_nombre_norm,scvs_nombre_raw,score,status,ruc,nombre_oficial_match'
  len : 83
[CHECK] scvs_matches_threshold_85_clean.csv
  repr: 'empresa_norm,scvs_nombre_norm,scvs_nombre_raw,score,status,ruc,nombre_oficial_match'
  len : 83


In [15]:
# (C.3) Debug: path real y primer header con bytes
import os
from pathlib import Path

print('[CWD]', os.getcwd())

p = Path('outputs') / f"scvs_matches_threshold_{MATCH_THRESHOLD}.csv"
print('[PATH]', str(p.resolve()))

b = p.read_bytes()[:160]
print('[BYTES]', b)

# Mostrar primera línea tal cual (sin decode raro)
line = b.splitlines()[0].decode('utf-8-sig', errors='replace')
print('[LINE repr]', repr(line))
print('[HAS padding token " ,"]', ' ,' in line or ', ' in line)

[CWD] e:\TESIS MAESTRIA\Desarrollo_clustering_maestria\01_data_ingestion_enrichment
[PATH] E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\01_data_ingestion_enrichment\outputs\scvs_matches_threshold_85.csv
[BYTES] b'\xef\xbb\xbfempresa_norm,scvs_nombre_norm,scvs_nombre_raw,score,status,ruc,nombre_oficial_match\r\n3DPHARMA,593PHARMA S A S,593PHARMA S.A.S.,66.66666666666666,below_thresh'
[LINE repr] 'empresa_norm,scvs_nombre_norm,scvs_nombre_raw,score,status,ruc,nombre_oficial_match'
[HAS padding token " ,"] False


In [16]:
# (D) Export final: empresas + RUC y descarga
from pathlib import Path
from typing import Optional, Sequence

import pandas as pd

outputs_dir = Path("outputs")
outputs_dir.mkdir(parents=True, exist_ok=True)

try:
    match_threshold = MATCH_THRESHOLD  # noqa: F821
except NameError:
    match_threshold = 85

raw_path = outputs_dir / f"scvs_matches_threshold_{match_threshold}.csv"
clean_path = raw_path.with_name(raw_path.stem + "_clean" + raw_path.suffix)

candidates = [p for p in (clean_path, raw_path) if p.exists()]
if not candidates:
    candidates = sorted(outputs_dir.glob("*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError(f"No hay archivos .csv en: {outputs_dir.resolve()}")

latest_file = max(candidates, key=lambda p: p.stat().st_mtime)
print("[Último archivo]", latest_file.name)
print("[Ruta]", latest_file.resolve())

df = pd.read_csv(latest_file, dtype=str)

def _pick_col(df_: pd.DataFrame, preferred: Sequence[str], contains_any: Optional[Sequence[str]] = None) -> str:
    cols = list(df_.columns)
    lower_map = {c.lower().strip(): c for c in cols}
    for p in preferred:
        if p in cols:
            return p
        p_norm = p.lower().strip()
        if p_norm in lower_map:
            return lower_map[p_norm]
    if contains_any:
        for c in cols:
            c_norm = c.lower().strip()
            if any(tok in c_norm for tok in contains_any):
                return c
    raise KeyError(f"No pude inferir columna. Columnas disponibles: {cols}")

col_empresa = _pick_col(
    df,
    preferred=["empresa", "empresa_norm", "empresas", "company", "company_name", "razon_social", "razón_social", "nombre_empresa"],
    contains_any=["empresa", "company", "razon", "razón", "nombre"],
)
col_ruc = _pick_col(df, preferred=["ruc", "numero_ruc", "nro_ruc"], contains_any=["ruc"])

out_path = outputs_dir / "empresasruc.csv"
out_df = df[[col_empresa, col_ruc]].copy()
out_df.columns = ["empresa", "ruc"]
out_df["ruc"] = out_df["ruc"].astype(str).str.replace(".0", "", regex=False).str.strip()
out_df["empresa"] = out_df["empresa"].astype(str).str.strip()
out_df = out_df.dropna(subset=["empresa", "ruc"]).drop_duplicates()

out_df.to_csv(out_path, index=False, encoding="utf-8-sig")
print("[Generado]", out_path.name, "| filas:", len(out_df))

display(out_df.head(20))

# Re-descargar/descargar (Colab) o dejar links (Jupyter local)
try:
    from google.colab import files  # type: ignore
    files.download(str(latest_file))
    files.download(str(out_path))
except Exception:
    from IPython.display import FileLink, display  # type: ignore
    display(FileLink(str(latest_file)))
    display(FileLink(str(out_path)))

[Último archivo] scvs_matches_threshold_85_clean.csv
[Ruta] E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\01_data_ingestion_enrichment\outputs\scvs_matches_threshold_85_clean.csv
[Generado] empresasruc.csv | filas: 447


,empresa,ruc
0,3DPHARMA,1891811969001
1,AVIS,1793119549001
2,ADIUM,0993265837001
3,ALMEXA,1793214892001
4,ALPER SEGUROS,1790525201001
5,ARAUCO,0992332964001
6,ASEGURADORA DEL SUR,0190123626001
7,ASESORIA Y CONTROL,0992698756001
8,AUTOSHARE,1792231116001
9,BAC,0992281707001


e:\TESIS MAESTRIA\Desarrollo_clustering_maestria\01_data_ingestion_enrichment\outputs\scvs_matches_threshold_85_clean.csv

e:\TESIS MAESTRIA\Desarrollo_clustering_maestria\01_data_ingestion_enrichment\outputs\empresasruc.csv